## DAR054 - Preventing serious complications in patients with haematological malignancies
Partner: Pulse AI (Commercial)

Purpose: ML model development for predicting complications (neutropenic sepsis, TLS, CRS, etc.)

Cohort: Patients with specified haematological malignancy ICD-10 codes (C81-C96, D46).
Subcohort view provides CATEGORY, TIER, and SUBTYPE breakdowns.

## Configuration

In [0]:
project_identifier = 'dar054'

# IG thresholds: columns must have BOTH ig_risk <= max_ig_risk AND ig_severity <= max_ig_severity to be included.
# Untagged columns are excluded unless listed in columns_to_include.
# max_ig_risk = 3 excludes: MRN, NHS_Number, Postcode, DOB, DOD, raw BlobContents
# max_ig_severity = 2 excludes: unanonymized free text
max_ig_risk = 3
max_ig_severity = 2

# Always exclude these columns regardless of tags.
columns_to_exclude = ['ADC_UPDT']

# Always include these columns regardless of tags (e.g. untagged columns you've manually reviewed).
# Format: {'catalog.schema.table': ['col1', 'col2']} or use '*' as table key for all tables.
columns_to_include = {}

# Tables that should bypass IG filtering entirely (e.g. reference/vocabulary tables with no patient data).
# These get SELECT * with no column filtering. Still cohort-filtered if a person_id column exists.
unfiltered_tables = []

# ---------------------------------------------------------------------------
# Table lists
# ---------------------------------------------------------------------------

# RDE tables (from 4_prod.rde)
rde_tables = [
    # Demographics
    'rde_patient_demographics',
    # Diagnoses
    'rde_all_diagnosis',
    'rde_all_problems',
    # Procedures
    'rde_all_procedures',
    # Encounters
    'rde_encounter',
    'rde_emergencyd',
    'rde_emergency_findings',
    # Labs/Measurements
    'rde_pathology',
    'rde_raw_pathology',
    'rde_measurements',
    # Medications
    'rde_medadmin',
    'rde_pharmacyorders',
    'rde_ariapharmacy',
    'rde_iqemo',
    # Critical Care
    'rde_critperiod',
    'rde_critactivity',
    'rde_critopcs',
    # Radiology
    'rde_radiology',
    # Free Text
    'rde_blobdataset',
    # Additional clinical data
    'rde_allergydetails',
    'rde_family_history',
    'rde_mill_powertrials',
    # Reference
    'rde_powerforms',
]

# Bronze/map tables (from 4_prod.bronze)
map_tables = ['map_patient_journey']

# OMOP tables (from 4_prod.omop)
omop_tables = []

# Raw/mill tables (from 4_prod.raw)
mill_tables = []

## Cohort

In [0]:
cohort_codes = [
    # Lymphoma (C81-C85) - Tier 1
    'C835', 'C844', 'C833', 'C831', 'C851', 'C859', 'C817',
    # Lymphoma (C81-C85) - Tier 2
    'C823', 'C830', 'C822', 'C821', 'C820', 'C840',
    'C810', 'C811', 'C812', 'C813', 'C814',
    # AML (C92) - Tier 1
    'C924', 'C926', 'C925', 'C920',
    # AML (C92) - Tier 2
    'C928', 'C929', 'C923', 'C927',
    # Multiple Myeloma (C90) - Tier 1
    'C902', 'C900', 'C909',
    # Multiple Myeloma (C90) - Tier 2
    'C901', 'C903',
    # MDS (D46) - Tier 1
    'D462', 'D469',
    # MDS (D46) - Tier 2
    'D460', 'D461', 'D464',
    # CLL/Lymphoid (C91) - Tier 1
    'C912', 'C913', 'C910',
    # CLL/Lymphoid (C91) - Tier 2
    'C911', 'C914', 'C915', 'C917', 'C919',
    # CML (C92.1-C92.2) - Tier 1
    'C922',
    # CML (C92.1-C92.2) - Tier 2
    'C921',
]

cohort_codes_sql = ", ".join([f"'{code}'" for code in cohort_codes])

cohort_sql = f"""
CREATE OR REPLACE VIEW 6_mgmt.cohorts.{project_identifier} AS
SELECT DISTINCT Person_ID AS PERSON_ID
FROM 4_prod.bronze.map_diagnosis
WHERE (
    SUBSTRING(REPLACE(ICD10_CODE, '.', ''), 1, 5) IN ({cohort_codes_sql})
    OR SUBSTRING(REPLACE(ICD10_CODE, '.', ''), 1, 4) IN ({cohort_codes_sql})
)
"""
spark.sql(cohort_sql)
print(f"Created cohort view: 6_mgmt.cohorts.{project_identifier}")

cohort_stats = spark.sql(f"""
    SELECT COUNT(DISTINCT PERSON_ID) as total_patients
    FROM 6_mgmt.cohorts.{project_identifier}
""")
display(cohort_stats)

## Schema Setup

In [0]:
spark.sql("USE CATALOG 5_projects")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS 5_projects.{project_identifier}")

existing_views_df = spark.sql(f"SHOW VIEWS IN 5_projects.{project_identifier}")
if existing_views_df.count() > 0:
    for row in existing_views_df.collect():
        view_name = row.viewName
        spark.sql(f"DROP VIEW IF EXISTS 5_projects.{project_identifier}.{view_name}")
        print(f"Dropped view: 5_projects.{project_identifier}.{view_name}")

# Also drop any existing tables (e.g. materialised results from previous runs)
existing_tables_df = spark.sql(f"SHOW TABLES IN 5_projects.{project_identifier}")
existing_views_set = set()
if existing_views_df.count() > 0:
    existing_views_set = {row.viewName for row in existing_views_df.collect()}
for row in existing_tables_df.collect():
    if row.tableName not in existing_views_set and row.tableName != project_identifier:
        spark.sql(f"DROP TABLE IF EXISTS 5_projects.{project_identifier}.{row.tableName}")
        print(f"Dropped table: 5_projects.{project_identifier}.{row.tableName}")

## Helper Functions

In [0]:
def get_safe_columns(catalog, schema, table):
    """
    Return a list of column names that pass IG filtering for the given table.

    A column is included only if:
      1. It has ig_risk <= max_ig_risk AND ig_severity <= max_ig_severity, OR
      2. It is in the columns_to_include override for this table.

    A column is excluded if:
      - It has no ig_risk/ig_severity tags (treated as max risk), OR
      - Its tag values exceed the thresholds, OR
      - It is in columns_to_exclude.
    """
    full_path = f"{catalog}.{schema}.{table}"
    all_columns = spark.table(full_path).columns

    # Get columns that are within both IG thresholds (must have BOTH tags within limits)
    safe_df = spark.sql(f"""
        SELECT r.column_name
        FROM (
            SELECT column_name, CAST(tag_value AS INT) AS risk_val
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}'
              AND table_name = '{table}'
              AND tag_name = 'ig_risk'
        ) r
        JOIN (
            SELECT column_name, CAST(tag_value AS INT) AS sev_val
            FROM {catalog}.information_schema.column_tags
            WHERE schema_name = '{schema}'
              AND table_name = '{table}'
              AND tag_name = 'ig_severity'
        ) s ON r.column_name = s.column_name
        WHERE r.risk_val <= {max_ig_risk}
          AND s.sev_val <= {max_ig_severity}
    """)
    safe_columns = set(safe_df.toPandas()['column_name'].tolist())

    # Add any explicit includes for this table
    includes = set(columns_to_include.get(full_path, []))
    includes |= set(columns_to_include.get('*', []))
    safe_columns |= includes

    # Remove explicit excludes
    safe_columns -= set(columns_to_exclude)

    # Preserve original column order
    return [c for c in all_columns if c in safe_columns]


def find_person_id_column(catalog, schema, table):
    """Find the person ID column in a table, checking common name variations."""
    full_path = f"{catalog}.{schema}.{table}"
    columns = spark.table(full_path).columns

    for candidate in ['PERSON_ID', 'person_id', 'Person_ID', 'PERSONID', 'PersonID', 'participant_id']:
        if candidate in columns:
            return candidate

    for col in columns:
        if 'person' in col.lower() and 'id' in col.lower():
            return col

    return None


def create_cohort_filtered_view(catalog, schema, table, project_id, column_list=None, alias=None):
    """
    Create a cohort-filtered view in 5_projects.{project_id}.

    Args:
        catalog: Source catalog (e.g. '4_prod')
        schema: Source schema (e.g. 'rde', 'bronze', 'omop')
        table: Source table name
        project_id: Project identifier for the target schema and cohort
        column_list: Explicit list of columns to select. If None, uses get_safe_columns().
                     Pass ['*'] to select all columns (for unfiltered reference tables).
        alias: Optional view name override. Defaults to the source table name.
    """
    full_path = f"{catalog}.{schema}.{table}"
    view_name = alias or table

    if column_list is None:
        column_list = get_safe_columns(catalog, schema, table)
        if not column_list:
            print(f"WARNING: No columns passed IG filter for {full_path}. Skipping.")
            return False

    person_id_col = find_person_id_column(catalog, schema, table)

    if column_list == ['*']:
        columns_sql = "t.*"
    else:
        columns_sql = ", ".join([f"t.`{c}`" for c in column_list])

    if person_id_col:
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        INNER JOIN 6_mgmt.cohorts.{project_id} c
            ON t.{person_id_col} = c.PERSON_ID
        """
    else:
        print(f"Note: No person ID column in {full_path}. Creating view without cohort filtering.")
        view_sql = f"""
        CREATE OR REPLACE VIEW 5_projects.{project_id}.{view_name} AS
        SELECT {columns_sql}
        FROM {full_path} t
        """

    spark.sql(view_sql)
    print(f"Created view: 5_projects.{project_id}.{view_name} ({len(column_list) if column_list != ['*'] else 'all'} columns)")
    return True

## Create Views

In [0]:
created = []
failed = []

# --- RDE tables ---
for table in rde_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'rde', table, project_identifier):
            created.append(f"rde.{table}")
    except Exception as e:
        failed.append((f"rde.{table}", str(e)))
        print(f"ERROR: rde.{table}: {e}")

# --- Bronze/map tables ---
for table in map_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'bronze', table, project_identifier):
            created.append(f"bronze.{table}")
    except Exception as e:
        failed.append((f"bronze.{table}", str(e)))
        print(f"ERROR: bronze.{table}: {e}")

# --- OMOP tables ---
for table in omop_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'omop', table, project_identifier):
            created.append(f"omop.{table}")
    except Exception as e:
        failed.append((f"omop.{table}", str(e)))
        print(f"ERROR: omop.{table}: {e}")

# --- Raw/mill tables ---
for table in mill_tables:
    try:
        if create_cohort_filtered_view('4_prod', 'raw', table, project_identifier):
            created.append(f"raw.{table}")
    except Exception as e:
        failed.append((f"raw.{table}", str(e)))
        print(f"ERROR: raw.{table}: {e}")

# --- Unfiltered reference tables (bypass IG, still cohort-filtered if person_id exists) ---
for table in unfiltered_tables:
    try:
        if '.' in table:
            parts = table.split('.')
            cat, sch, tbl = parts[0], parts[1], parts[2]
        else:
            cat, sch, tbl = '4_prod', 'omop', table
        if create_cohort_filtered_view(cat, sch, tbl, project_identifier, column_list=['*']):
            created.append(f"{sch}.{tbl}")
    except Exception as e:
        failed.append((f"{table}", str(e)))
        print(f"ERROR: {table}: {e}")

## Bespoke Views
### Subcohort Lookup
Maps each patient to their haematological malignancy ICD-10 categories.
Patients may have multiple subcohorts if they have multiple qualifying diagnoses.

In [0]:
subcohort_sql = f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.subcohort AS
WITH coded_diagnoses AS (
    SELECT
        PERSON_ID,
        REPLACE(ICD10_CODE, '.', '') AS Diagnosis_code,
        SUBSTRING(REPLACE(ICD10_CODE, '.', ''), 1, 5) AS code_5char,
        SUBSTRING(REPLACE(ICD10_CODE, '.', ''), 1, 4) AS code_4char,
        DIAG_DT_TM AS DIAGNOSIS_DATE
    FROM 4_prod.bronze.map_diagnosis
    WHERE Person_ID IN (SELECT PERSON_ID FROM 6_mgmt.cohorts.{project_identifier})
),
categorized AS (
    SELECT
        PERSON_ID,
        Diagnosis_code,
        DIAGNOSIS_DATE,
        -- Determine SUBTYPE (the specific code)
        CASE
            WHEN code_5char IN ('C835','C844','C833','C831','C851','C859','C817',
                                'C823','C830','C822','C821','C820','C840',
                                'C810','C811','C812','C813','C814') THEN code_5char
            WHEN code_5char IN ('C924','C926','C925','C920','C928','C929','C923','C927','C922','C921') THEN code_5char
            WHEN code_5char IN ('C902','C900','C909','C901','C903') THEN code_5char
            WHEN code_5char IN ('D462','D469','D460','D461','D464') THEN code_5char
            WHEN code_5char IN ('C912','C913','C910','C911','C914','C915','C917','C919') THEN code_5char
            WHEN code_4char IN ('C835','C844','C833','C831','C851','C859','C817',
                                'C823','C830','C822','C821','C820','C840',
                                'C810','C811','C812','C813','C814') THEN code_4char
            WHEN code_4char IN ('C924','C926','C925','C920','C928','C929','C923','C927','C922','C921') THEN code_4char
            WHEN code_4char IN ('C902','C900','C909','C901','C903') THEN code_4char
            WHEN code_4char IN ('D462','D469','D460','D461','D464') THEN code_4char
            WHEN code_4char IN ('C912','C913','C910','C911','C914','C915','C917','C919') THEN code_4char
            ELSE NULL
        END AS SUBTYPE,
        -- Determine CATEGORY
        CASE
            WHEN code_5char LIKE 'C81%' OR code_5char LIKE 'C82%' OR code_5char LIKE 'C83%'
                 OR code_5char LIKE 'C84%' OR code_5char LIKE 'C85%'
                 OR code_4char LIKE 'C81%' OR code_4char LIKE 'C82%' OR code_4char LIKE 'C83%'
                 OR code_4char LIKE 'C84%' OR code_4char LIKE 'C85%' THEN 'Lymphoma'
            WHEN (code_5char IN ('C924','C926','C925','C920','C928','C929','C923','C927')
                  OR code_4char IN ('C924','C926','C925','C920','C928','C929','C923','C927')) THEN 'AML'
            WHEN (code_5char IN ('C921','C922') OR code_4char IN ('C921','C922')) THEN 'CML'
            WHEN code_5char LIKE 'C90%' OR code_4char LIKE 'C90%' THEN 'Multiple Myeloma'
            WHEN code_5char LIKE 'D46%' OR code_4char LIKE 'D46%' THEN 'MDS'
            WHEN code_5char LIKE 'C91%' OR code_4char LIKE 'C91%' THEN 'CLL/Lymphoid Leukaemia'
            ELSE NULL
        END AS CATEGORY,
        -- Determine TIER
        CASE
            WHEN code_5char IN ('C835','C844','C833','C831','C851','C859','C817')
                 OR code_4char IN ('C835','C844','C833','C831','C851','C859','C817') THEN 'Tier 1'
            WHEN code_5char IN ('C823','C830','C822','C821','C820','C840','C810','C811','C812','C813','C814')
                 OR code_4char IN ('C823','C830','C822','C821','C820','C840','C810','C811','C812','C813','C814') THEN 'Tier 2'
            WHEN code_5char IN ('C924','C926','C925','C920')
                 OR code_4char IN ('C924','C926','C925','C920') THEN 'Tier 1'
            WHEN code_5char IN ('C928','C929','C923','C927')
                 OR code_4char IN ('C928','C929','C923','C927') THEN 'Tier 2'
            WHEN code_5char IN ('C902','C900','C909')
                 OR code_4char IN ('C902','C900','C909') THEN 'Tier 1'
            WHEN code_5char IN ('C901','C903')
                 OR code_4char IN ('C901','C903') THEN 'Tier 2'
            WHEN code_5char IN ('D462','D469')
                 OR code_4char IN ('D462','D469') THEN 'Tier 1'
            WHEN code_5char IN ('D460','D461','D464')
                 OR code_4char IN ('D460','D461','D464') THEN 'Tier 2'
            WHEN code_5char IN ('C912','C913','C910')
                 OR code_4char IN ('C912','C913','C910') THEN 'Tier 1'
            WHEN code_5char IN ('C911','C914','C915','C917','C919')
                 OR code_4char IN ('C911','C914','C915','C917','C919') THEN 'Tier 2'
            WHEN code_5char = 'C922' OR code_4char = 'C922' THEN 'Tier 1'
            WHEN code_5char = 'C921' OR code_4char = 'C921' THEN 'Tier 2'
            ELSE NULL
        END AS TIER
    FROM coded_diagnoses
)
SELECT
    PERSON_ID,
    CATEGORY,
    TIER,
    SUBTYPE,
    Diagnosis_code AS DIAGNOSIS_CODE,
    MIN(DIAGNOSIS_DATE) AS FIRST_DIAGNOSIS_DATE
FROM categorized
WHERE SUBTYPE IS NOT NULL
GROUP BY PERSON_ID, CATEGORY, TIER, SUBTYPE, Diagnosis_code
ORDER BY PERSON_ID, CATEGORY, TIER, FIRST_DIAGNOSIS_DATE
"""
spark.sql(subcohort_sql)
created.append("subcohort")
print(f"Created subcohort view: 5_projects.{project_identifier}.subcohort")

display(spark.sql(f"""
    SELECT
        CATEGORY,
        TIER,
        SUBTYPE,
        COUNT(DISTINCT PERSON_ID) as patient_count
    FROM 5_projects.{project_identifier}.subcohort
    GROUP BY CATEGORY, TIER, SUBTYPE
    ORDER BY CATEGORY, TIER, SUBTYPE
"""))

## Schema Documentation View

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW 5_projects.{project_identifier}.schema AS
SELECT
    table_name,
    column_name,
    COALESCE(comment, '') AS column_comment
FROM 5_projects.information_schema.columns
WHERE table_catalog = '5_projects'
  AND table_schema = '{project_identifier}'
  AND table_name != 'schema'
ORDER BY table_name, column_name
""")
print(f"Created schema view: 5_projects.{project_identifier}.schema")

## Summary

In [0]:
print("=" * 60)
print(f"Project Setup Complete: {project_identifier}")
print("=" * 60)
print(f"\nCohort:  6_mgmt.cohorts.{project_identifier}")
print(f"Schema:  5_projects.{project_identifier}")
print(f"\nIG Thresholds: ig_risk <= {max_ig_risk}, ig_severity <= {max_ig_severity}")
print(f"Policy: untagged columns are EXCLUDED (treated as max risk)")
print(f"\nViews created: {len(created)}")
for v in created:
    print(f"  + {v}")
if failed:
    print(f"\nFailed: {len(failed)}")
    for t, e in failed:
        print(f"  x {t}: {e}")

## Verification Queries

In [0]:
print("1. Cohort size:")
display(spark.sql(f"SELECT COUNT(DISTINCT PERSON_ID) as cohort_size FROM 6_mgmt.cohorts.{project_identifier}"))

In [0]:
print("2. Views in project schema:")
display(spark.sql(f"SHOW VIEWS IN 5_projects.{project_identifier}"))

In [0]:
print("3. Subcohort distribution:")
display(spark.sql(f"""
    SELECT
        SUBTYPE,
        COUNT(DISTINCT PERSON_ID) as patients,
        COUNT(*) as diagnosis_records
    FROM 5_projects.{project_identifier}.subcohort
    GROUP BY SUBTYPE
    ORDER BY SUBTYPE
"""))